In [ ]:
!pip install -q torch torchvision scikit-learn pandas matplotlib seaborn pillow

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# CHANGE ONLY THIS PATH if your dataset is stored somewhere else
BASE_DIR = Path("/content/drive/MyDrive/capstone_dataset creation/capstone_dataset creation")

ORIGINAL_DIR = BASE_DIR / "original"
TAMPERED_DIR = BASE_DIR / "tampered"
METADATA_DIR = BASE_DIR / "metadata"

CSV_PATH = METADATA_DIR / "master_dataset.csv"

print("Dataset directory:", BASE_DIR)
print("CSV:", CSV_PATH)
print("Original folder:", ORIGINAL_DIR)
print("Tampered folder:", TAMPERED_DIR)

In [ ]:
assert BASE_DIR.exists(), f"Dataset directory not found: {BASE_DIR}"
assert ORIGINAL_DIR.exists(), f"Original folder not found: {ORIGINAL_DIR}"
assert TAMPERED_DIR.exists(), f"Tampered folder not found: {TAMPERED_DIR}"
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"

print("Dataset paths verified successfully.")

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("Number of records:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

In [ ]:
print("Label distribution:")
print(df["label"].value_counts())

print("\nExpected:")
print("Original :", 9)
print("Tampered :", 567)

In [ ]:
# Check how many records belong to each certificate
group_counts = df.groupby("certificate_id").size()

print(group_counts)

print("\nNumber of certificate groups:", len(group_counts))

In [ ]:
from sklearn.model_selection import train_test_split

# Get the 9 unique certificate groups
certificate_groups = sorted(df["certificate_id"].unique())

print("Certificate groups:")
for group in certificate_groups:
    print(group)

# 6 groups for training, 3 remaining
train_groups, remaining_groups = train_test_split(
    certificate_groups,
    test_size=3,
    random_state=42
)

# Of the remaining 3:
# 2 validation groups
# 1 test group
val_groups, test_groups = train_test_split(
    remaining_groups,
    test_size=1,
    random_state=42
)

print("\nTRAIN groups:", train_groups)
print("VALIDATION groups:", val_groups)
print("TEST groups:", test_groups)

In [ ]:
train_df = df[df["certificate_id"].isin(train_groups)].copy()
val_df = df[df["certificate_id"].isin(val_groups)].copy()
test_df = df[df["certificate_id"].isin(test_groups)].copy()

print("TRAIN")
print("Records:", len(train_df))
print(train_df["label"].value_counts())

print("\nVALIDATION")
print("Records:", len(val_df))
print(val_df["label"].value_counts())

print("\nTEST")
print("Records:", len(test_df))
print(test_df["label"].value_counts())

In [ ]:
train_set = set(train_df["certificate_id"])
val_set = set(val_df["certificate_id"])
test_set = set(test_df["certificate_id"])

print("Train ∩ Validation:", train_set.intersection(val_set))
print("Train ∩ Test:", train_set.intersection(test_set))
print("Validation ∩ Test:", val_set.intersection(test_set))

assert len(train_set.intersection(val_set)) == 0
assert len(train_set.intersection(test_set)) == 0
assert len(val_set.intersection(test_set)) == 0

print("\nPASS: No certificate appears in more than one split.")

In [ ]:
def get_image_path(row):
    image_name = str(row["image"])

    if row["label"] == "Original":
        return str(ORIGINAL_DIR / image_name)
    else:
        return str(TAMPERED_DIR / image_name)


train_df["image_path"] = train_df.apply(get_image_path, axis=1)
val_df["image_path"] = val_df.apply(get_image_path, axis=1)
test_df["image_path"] = test_df.apply(get_image_path, axis=1)

print(train_df[["image", "label", "image_path"]].head())

In [ ]:
def check_images(dataframe, name):
    missing = [
        path for path in dataframe["image_path"]
        if not Path(path).exists()
    ]

    print(f"{name}: {len(dataframe)} records")
    print(f"{name}: {len(missing)} missing images")

    if missing:
        print("First missing files:")
        for path in missing[:10]:
            print(path)

    return missing


train_missing = check_images(train_df, "TRAIN")
val_missing = check_images(val_df, "VALIDATION")
test_missing = check_images(test_df, "TEST")

assert len(train_missing) == 0
assert len(val_missing) == 0
assert len(test_missing) == 0

print("\nPASS: All required images exist.")

In [ ]:
LABEL_MAP = {
    "Original": 0,
    "Tampered": 1
}

train_df["target"] = train_df["label"].map(LABEL_MAP)
val_df["target"] = val_df["label"].map(LABEL_MAP)
test_df["target"] = test_df["label"].map(LABEL_MAP)

print(train_df[["label", "target"]].drop_duplicates())

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from PIL import Image
import numpy as np

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=3),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class CertificateDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = int(row["target"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = CertificateDataset(
    train_df,
    transform=train_transform
)

val_dataset = CertificateDataset(
    val_df,
    transform=eval_transform
)

test_dataset = CertificateDataset(
    test_df,
    transform=eval_transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
class_counts = train_df["target"].value_counts().sort_index()

print("Class counts:")
print(class_counts)

class_weights = {
    class_id: 1.0 / count
    for class_id, count in class_counts.items()
}

sample_weights = train_df["target"].map(class_weights).values

sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\nWeighted sampler created.")

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created.")

In [ ]:
model = efficientnet_b0(weights=weights)

# Replace the final classification layer
num_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    num_features,
    2
)

model = model.to(device)

print(model.classifier)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss, optimizer and scheduler ready.")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def evaluate_model(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return (
        epoch_loss,
        epoch_accuracy,
        all_labels,
        all_predictions
    )

In [ ]:
NUM_EPOCHS = 10

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss = float("inf")

MODEL_SAVE_PATH = "/content/drive/MyDrive/efficientnet_b0_certificate_tampering_best.pth"

for epoch in range(NUM_EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_accuracy, _, _ = evaluate_model(
        model,
        val_loader,
        criterion
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)

    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_mapping": LABEL_MAP,
                "model_name": "EfficientNet-B0"
            },
            MODEL_SAVE_PATH
        )

        print("  → Best model saved.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(history["train_accuracy"], label="Training Accuracy")
plt.plot(history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history["train_loss"], label="Training Loss")
plt.plot(history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
checkpoint = torch.load(
    MODEL_SAVE_PATH,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)

print("Best model loaded successfully.")

In [ ]:
test_loss, test_accuracy, test_labels, test_predictions = evaluate_model(
    model,
    test_loader,
    criterion
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
from sklearn.metrics import classification_report

print("=" * 35)
print("FINAL TEST RESULTS")
print("=" * 35)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        test_labels,
        test_predictions,
        labels=[0, 1],
        target_names=["Original", "Tampered"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(
    test_labels,
    test_predictions
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Original", "Tampered"],
    yticklabels=["Original", "Tampered"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
def predict_certificate(image_path, model):

    model.eval()

    image = Image.open(image_path).convert("RGB")

    image_tensor = eval_transform(image)
    image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.to(device)

    with torch.no_grad():

        output = model(image_tensor)

        probabilities = torch.softmax(output, dim=1)

        confidence, predicted_class = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = predicted_class.item()
    confidence = confidence.item()

    class_names = {
        0: "Original",
        1: "Tampered"
    }

    return (
        class_names[predicted_class],
        confidence
    )

In [ ]:
sample_path = test_df.iloc[0]["image_path"]

prediction, confidence = predict_certificate(
    sample_path,
    model
)

print("Image:", sample_path)
print("Prediction:", prediction)
print("Confidence:", f"{confidence * 100:.2f}%")

In [ ]:
FINAL_MODEL_PATH = "/content/drive/MyDrive/efficientnet_b0_certificate_tampering_FINAL.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_mapping": {
            0: "Original",
            1: "Tampered"
        },
        "model_name": "EfficientNet-B0",
        "input_size": 224
    },
    FINAL_MODEL_PATH
)

print("Final model saved to:")
print(FINAL_MODEL_PATH)

In [ ]:
# ============================================================
# CELL 31 — CLASS MAPPING
# ============================================================

CLASS_NAMES = {
    0: "Original",
    1: "Tampered"
}

print("Class mapping:")
print("0 =", CLASS_NAMES[0])
print("1 =", CLASS_NAMES[1])

In [ ]:
# ============================================================
# CELL 32 — LOAD FINAL EFFICIENTNET-B0 MODEL
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "/content/drive/MyDrive/efficientnet_b0_certificate_tampering_FINAL.pth"

model = models.efficientnet_b0(weights=None)

# Same classifier structure used during training
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    2
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

# Handle the checkpoint format used when saving
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model = model.to(DEVICE)
model.eval()

print("Model loaded successfully.")
print("Device:", DEVICE)
print("Model: EfficientNet-B0")

In [ ]:
# ============================================================
# CELL 33 — INFERENCE PREPROCESSING
# ============================================================

from torchvision import transforms

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Inference preprocessing ready.")

In [ ]:
# ============================================================
# CELL 34 — CERTIFICATE PREDICTION FUNCTION
# ============================================================

from PIL import Image

def predict_certificate(image_path):
    """
    Predict whether a certificate is Original or Tampered.

    Returns:
        prediction: Original or Tampered
        confidence: confidence percentage
    """

    image = Image.open(image_path).convert("RGB")

    image_tensor = inference_transform(image)
    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image_tensor)

        probabilities = torch.softmax(outputs, dim=1)

        confidence, predicted_class = torch.max(
            probabilities, dim=1
        )

    predicted_class = predicted_class.item()
    confidence = confidence.item() * 100

    prediction = CLASS_NAMES[predicted_class]

    return prediction, confidence

In [ ]:
import os
from pathlib import Path

ORIGINAL_DIR = Path("/content/drive/MyDrive/capstone_dataset creation/capstone_dataset creation/original")

if ORIGINAL_DIR.exists():
    print("Files in original folder:")
    for filename in os.listdir(ORIGINAL_DIR):
        print(filename)
else:
    print(f"Directory does not exist: {ORIGINAL_DIR}")
    parent_dir = ORIGINAL_DIR.parent
    if parent_dir.exists():
        print(f"\nContents of parent directory ({parent_dir}):")
        for item in os.listdir(parent_dir):
            print(f" - {item}")
    else:
        print(f"\nParent directory also does not exist: {parent_dir}")

In [ ]:
# ============================================================
# CELL 35 — TEST PREDICTION
# ============================================================

TEST_IMAGE = "/content/drive/MyDrive/capstone_dataset creation/capstone_dataset creation/original/image_original_as_2.jpeg"

prediction, confidence = predict_certificate(TEST_IMAGE)

print("=" * 50)
print("CERTIFICATE PREDICTION")
print("=" * 50)
print("Prediction :", prediction)
print(f"Confidence : {confidence:.2f}%")

In [ ]:
# ============================================================
# TEST TAMPERED CERTIFICATE
# ============================================================

import os

TAMPERED_DIR = "/content/drive/MyDrive/capstone_dataset creation/capstone_dataset creation/tampered"

print("Tampered files:")
for filename in os.listdir(TAMPERED_DIR)[:10]:
    print(filename)

In [ ]:
# ============================================================
# TEST TAMPERED CERTIFICATE
# ============================================================

TEST_TAMPERED_IMAGE = "/content/drive/MyDrive/capstone_dataset creation/capstone_dataset creation/tampered/image_original_1_name_test.jpeg"

prediction, confidence = predict_certificate(TEST_TAMPERED_IMAGE)

print("=" * 50)
print("TAMPERED CERTIFICATE PREDICTION")
print("=" * 50)
print("Prediction :", prediction)
print(f"Confidence : {confidence:.2f}%")